In [2]:
!pip install sam3 -q
!pip install huggingface_hub -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.2 MB/s eta 0:00:00


In [3]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
!pip install triton -q

In [6]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="facebook/sam3",
    local_dir="/content/sam3_weights"
)
print("다운로드 완료!")

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

다운로드 완료!


In [11]:
import os

os.makedirs("/usr/local/lib/python3.12/dist-packages/assets", exist_ok=True)

!wget -q "https://openaipublic.blob.core.windows.net/clip/bpe_simple_vocab_16e6.txt.gz" \
    -O "/usr/local/lib/python3.12/dist-packages/assets/bpe_simple_vocab_16e6.txt.gz"

print("다운로드 완료!")

다운로드 완료!


In [2]:
from google.colab import files
uploaded = files.upload()  #

Saving 1-1_bg.png to 1-1_bg.png
Saving 1-1_person.png to 1-1_person.png
Saving 1-2_bg.png to 1-2_bg.png
Saving 1-2_person.png to 1-2_person.png
Saving 1-3_bg.png to 1-3_bg.png
Saving 1-3_person.png to 1-3_person.png
Saving 2-1_bg.png to 2-1_bg.png
Saving 2-1_person.png to 2-1_person.png
Saving 2-2_bg.png to 2-2_bg.png
Saving 2-2_person.png to 2-2_person.png
Saving 2-3_bg.png to 2-3_bg.png
Saving 2-3_person.png to 2-3_person.png
Saving 3-1_bg.png to 3-1_bg.png
Saving 3-1_person.png to 3-1_person.png
Saving 3-2_bg.png to 3-2_bg.png
Saving 3-2_person.png to 3-2_person.png
Saving 3-3_bg.png to 3-3_bg.png
Saving 3-3_person.png to 3-3_person.png


In [3]:
import torch
from PIL import Image
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

# 모델 로드
model = build_sam3_image_model()
processor = Sam3Processor(model)

# 테스트 이미지
image = Image.open("1-1_person.png")
inference_state = processor.set_image(image)
output = processor.set_text_prompt(state=inference_state, prompt="person")

masks, boxes, scores = output["masks"], output["boxes"], output["scores"]
print(f"마스크 개수: {len(masks)}")

마스크 개수: 1


In [10]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

pairs = []
for i in range(1, 4):
    for j in range(1, 4):
        pairs.append(f"{i}-{j}")

for name in pairs:
    # 이미지 불러오기
    person_img = Image.open(f"{name}_person.png")
    bg_img = cv2.imread(f"{name}_bg.png")
    bg_img = cv2.cvtColor(bg_img, cv2.COLOR_BGR2RGB)

    # SAM3 추론
    inference_state = processor.set_image(person_img)
    output = processor.set_text_prompt(state=inference_state, prompt="person")
    masks = output["masks"]

    # 마스크 추출
    img_mask = masks[0].squeeze().cpu().numpy().astype(np.uint8) * 255

    # 원본 RGB
    cat_img = np.array(person_img.convert("RGB"))
    h, w = cat_img.shape[:2]

    # 마스크 크기 맞춤
    img_mask = cv2.resize(img_mask, (w, h))
    img_mask_color = cv2.cvtColor(img_mask, cv2.COLOR_GRAY2BGR)

    # 블러 합성
    blurred = cv2.GaussianBlur(cat_img, (101, 101), 0)
    result_blur = np.where(img_mask_color == 255, cat_img, blurred)

    # 배경 교체 합성
    bg_resized = cv2.resize(bg_img, (w, h))
    result_bg = np.where(img_mask_color == 255, cat_img, bg_resized)

    # 출력
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(cat_img)
    axes[0].set_title(f"[{name}] Original")
    axes[0].axis('off')

    axes[1].imshow(bg_resized)
    axes[1].set_title(f"[{name}] Background")
    axes[1].axis('off')

    axes[2].imshow(result_blur)
    axes[2].set_title(f"[{name}] Blur Result")
    axes[2].axis('off')

    axes[3].imshow(result_bg)
    axes[3].set_title(f"[{name}] BG Replace")
    axes[3].axis('off')

    plt.tight_layout()
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [8]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

# SAM3 모델 로드
model = build_sam3_image_model()
processor = Sam3Processor(model)
print("SAM3 로드 완료!")

# 파일 쌍 생성
pairs = []
for i in range(1, 4):
    for j in range(1, 4):
        pairs.append(f"{i}-{j}")

for name in pairs:
    # 이미지 불러오기
    person_img = Image.open(f"{name}_person.png")
    bg_img = cv2.imread(f"{name}_bg.png")
    bg_img = cv2.cvtColor(bg_img, cv2.COLOR_BGR2RGB)

    # SAM3 추론
    inference_state = processor.set_image(person_img)
    output = processor.set_text_prompt(state=inference_state, prompt="person")
    masks = output["masks"]
    boxes = output["boxes"]

    # 마스크 추출
    img_mask = masks[0].squeeze().cpu().numpy().astype(np.uint8) * 255

    # 원본 RGB
    cat_img = np.array(person_img.convert("RGB"))
    h, w = cat_img.shape[:2]
    img_mask = cv2.resize(img_mask, (w, h))

    # ✅ 후처리 - 구멍 메우기
    kernel = np.ones((20, 20), np.uint8)
    img_mask = cv2.morphologyEx(img_mask, cv2.MORPH_CLOSE, kernel)

    # ✅ 후처리 - Bounding Box로 마스크 보완
    if boxes is not None and len(boxes) > 0:
        x1, y1, x2, y2 = boxes[0].int().tolist()
        # 이미지 크기에 맞게 스케일 조정
        bh, bw = masks[0].squeeze().shape
        x1 = int(x1 * w / bw)
        x2 = int(x2 * w / bw)
        y1 = int(y1 * h / bh)
        y2 = int(y2 * h / bh)
        roi = img_mask[y1:y2, x1:x2]
        roi_filled = cv2.morphologyEx(
            roi, cv2.MORPH_CLOSE,
            np.ones((30, 30), np.uint8)
        )
        img_mask[y1:y2, x1:x2] = np.maximum(roi, roi_filled)

    img_mask_color = cv2.cvtColor(img_mask, cv2.COLOR_GRAY2BGR)

    # 블러 합성
    blurred = cv2.GaussianBlur(cat_img, (101, 101), 0)
    result_blur = np.where(img_mask_color == 255, cat_img, blurred)

    # 배경 교체 합성
    bg_resized = cv2.resize(bg_img, (w, h))
    result_bg = np.where(img_mask_color == 255, cat_img, bg_resized)

    # 출력
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(cat_img)
    axes[0].set_title(f"[{name}] Original")
    axes[0].axis('off')

    axes[1].imshow(bg_resized)
    axes[1].set_title(f"[{name}] Background")
    axes[1].axis('off')

    axes[2].imshow(result_blur)
    axes[2].set_title(f"[{name}] Blur Result")
    axes[2].axis('off')

    axes[3].imshow(result_bg)
    axes[3].set_title(f"[{name}] BG Replace")
    axes[3].axis('off')

    plt.tight_layout()
    plt.show()
    print(f"{name} 완료!")

Output hidden; open in https://colab.research.google.com to view.

# 전체적으로 DeepLabV3 → SAM3로 오면서 품질이 눈에 띄게 좋아졌지만 투명소재와 복잡한 배경에서는 완벽히 분리 되지 않음을 확인했다 모델 한계와 후처리의 중요성을 직접 체험한 좋은 실습이었음